# 📈 Training Analysis

This notebook analyzes the training results of the EEG-Transformer model and compares it to the EEGNet baseline.

We will cover:
1. Training & validation curves
2. Confusion matrix analysis
3. Statistical significance testing (Paired t-test)
4. Model comparison

In [1]:
import sys
import os
import json
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from src.visualization.training_curves import (
    plot_training_curves,
    plot_confusion_matrix,
    plot_model_comparison,
)
from src.training.metrics import (
    paired_ttest,
    compute_confidence_interval,
    cohens_d,
)

## 1. Training Curves

After training with `scripts/train_eeg.py`, results are saved to `assets/results/`. Here we demonstrate with synthetic data — replace with your actual training logs.

In [2]:
# Synthetic training curves for demonstration
np.random.seed(42)
epochs = 50
train_losses = 2.0 * np.exp(-np.arange(epochs) / 10) + 0.3 + 0.05 * np.random.randn(epochs)
val_losses = 2.0 * np.exp(-np.arange(epochs) / 12) + 0.5 + 0.08 * np.random.randn(epochs)
train_accs = 1.0 - train_losses / train_losses[0] + 0.03 * np.random.randn(epochs)
val_accs = 1.0 - val_losses / val_losses[0] + 0.05 * np.random.randn(epochs)

plot_training_curves(
    train_losses=train_losses.tolist(),
    val_losses=val_losses.tolist(),
    train_accs=np.clip(train_accs, 0, 1).tolist(),
    val_accs=np.clip(val_accs, 0, 1).tolist(),
    title="EEG-Transformer Training Curves (Synthetic Demo)",
)

## 2. Confusion Matrix

A confusion matrix reveals which classes the model confuses. For motor imagery, we often see confusion between left/right fist and between both_fists/both_feet.

In [3]:
from src.eeg.dataset import CLASS_NAMES

# Synthetic confusion matrix
cm = np.array([
    [35, 8, 3, 4],   # left_fist
    [10, 32, 4, 4],  # right_fist
    [2, 3, 38, 7],   # both_fists
    [3, 5, 6, 36],   # both_feet
])

plot_confusion_matrix(
    cm,
    class_names=CLASS_NAMES,
    title="EEG-Transformer Confusion Matrix (Synthetic Demo)",
    normalize=True,
)

## 3. Statistical Comparison: Transformer vs EEGNet

We use a **paired t-test** to determine if the accuracy difference between the Transformer and EEGNet baseline is statistically significant across subjects. We also compute **Cohen's d** for effect size.

In [4]:
# Simulated per-subject accuracy scores for 10 subjects
transformer_acc = [0.72, 0.81, 0.65, 0.78, 0.85, 0.69, 0.74, 0.79, 0.82, 0.71]
eegnet_acc = [0.68, 0.75, 0.61, 0.72, 0.79, 0.65, 0.70, 0.74, 0.76, 0.68]

# Paired t-test
ttest_results = paired_ttest(transformer_acc, eegnet_acc)
print("=" * 50)
print("PAIRED T-TEST: Transformer vs EEGNet")
print("=" * 50)
print(f"Mean Difference:  {ttest_results['mean_difference']:.4f}")
print(f"T-Statistic:      {ttest_results['t_statistic']:.4f}")
print(f"P-Value:          {ttest_results['p_value']:.4f}")
print(f"Significant:      {bool(ttest_results['is_significant'])}")

# Effect size
d = cohens_d(transformer_acc, eegnet_acc)
print(f"\nCohen's d:        {d:.4f} ({'small' if abs(d) < 0.5 else 'medium' if abs(d) < 0.8 else 'large'} effect)")

# Confidence intervals
for name, scores in [("Transformer", transformer_acc), ("EEGNet", eegnet_acc)]:
    mean, lo, hi = compute_confidence_interval(scores)
    print(f"\n{name}: {mean:.4f} (95% CI: [{lo:.4f}, {hi:.4f}])")

PAIRED T-TEST: Transformer vs EEGNet
Mean Difference:  0.0480
T-Statistic:      13.3701
P-Value:          0.0000
Significant:      True

Cohen's d:        0.8062 (large effect)

Transformer: 0.7560 (95% CI: [0.7102, 0.8018])

EEGNet: 0.7080 (95% CI: [0.6688, 0.7472])


## 4. Model Comparison

A bar chart comparing key metrics across models.

In [5]:
comparison_results = {
    "EEG-Transformer": {
        "accuracy": np.mean(transformer_acc),
        "accuracy_std": np.std(transformer_acc),
    },
    "EEGNet": {
        "accuracy": np.mean(eegnet_acc),
        "accuracy_std": np.std(eegnet_acc),
    },
}

plot_model_comparison(comparison_results, metric="accuracy")